# XGBoost: From First Principles

XGBoost (Extreme Gradient Boosting) is a regularized gradient-boosted tree system. It adds trees sequentially: each new tree is fitted to improve the loss of the model built so far, and the final prediction combines the contributions of all trees.

Its strength comes from both the boosting objective and an efficient implementation. The objective penalizes complexity; the implementation includes optimizations for split search, sparse data, and parallel computation. XGBoost is popular for structured data, but it is not automatically the best choice for every dataset or task.

## Why XGBoost Is Useful

Gradient boosting builds an additive model by fitting new learners to the remaining loss. It can be highly accurate, but unconstrained trees can overfit, and evaluating candidate splits across many rows and features can be expensive.

XGBoost addresses these concerns in two connected ways. Its objective explicitly penalizes tree size and leaf scores, while its implementation uses gradient statistics to score splits and engineering techniques such as sparsity-aware split handling, histogram-based methods, and parallel work where possible.

The name does not guarantee that it will outperform every other method. Results depend on data, validation design, feature representation, objective, and tuning. Simpler baselines such as linear models, Random Forest, or other boosting libraries remain useful comparisons.

## Additive Boosting: One Tree at a Time

Let $\hat{y}_i^{(t)}$ be the model's prediction for observation $i$ after round $t$. XGBoost adds a new tree's contribution to the current prediction:

$$
\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta f_t(\mathbf{x}_i)
$$

Here $f_t$ maps an observation to the score of the leaf it reaches, and $\eta$ is the learning rate (shrinkage). The tree is chosen to reduce the loss while controlling its complexity. The process repeats for a chosen number of rounds or until validation performance stops improving.

For squared-error regression, the gradient points in the direction of increasing prediction error, so fitting a tree to the negative gradient resembles fitting residuals. For other losses, such as logistic loss, the gradients provide a loss-aware direction and are not simply raw residuals.

## The Regularized Objective

At boosting round $t$, XGBoost adds a tree $f_t$ to the current predictions. Its objective combines the training loss with a penalty on the new tree:

$$
\mathcal{L}^{(t)} = \sum_{i=1}^{n} \ell\left(y_i,\hat{y}_i^{(t-1)}+f_t(\mathbf{x}_i)\right) + \Omega(f_t)
$$

A common tree penalty is:

$$
\Omega(f) = \gamma T + \frac{1}{2}\lambda\sum_{j=1}^{T}w_j^2 + \alpha\sum_{j=1}^{T}|w_j|
$$

Here $T$ is the number of leaves, $w_j$ is the score assigned to leaf $j$, $\gamma$ penalizes adding leaves, $\lambda$ applies L2 regularization to leaf scores, and $\alpha$ applies L1 regularization. Implementations may expose additional constraints and settings, but these terms show the core idea: fit the data while charging for tree complexity and large leaf values.

The training loss depends on all previous trees. At a boosting round, those existing predictions are fixed; the algorithm searches for an additional tree that improves the regularized objective.

## Second-Order Approximation

At round $t$, consider adding a tree output $f_t(\mathbf{x}_i)$ to the current prediction $\hat{y}_i^{(t-1)}$. XGBoost approximates each observation's loss with a second-order Taylor expansion around the current prediction:

$$
\ell(y_i,\hat{y}_i^{(t-1)}+f_t(\mathbf{x}_i)) \approx \ell(y_i,\hat{y}_i^{(t-1)}) + g_i f_t(\mathbf{x}_i) + \frac{1}{2}h_i f_t(\mathbf{x}_i)^2
$$

Here $g_i$ is the first derivative of the loss with respect to the current prediction, and $h_i$ is the second derivative. The first derivative indicates the local direction of loss change; the second derivative describes its local curvature. The previous loss term is constant while choosing the new tree, so the gradients and Hessians provide the statistics needed to compare candidate trees.

For a tree with leaf assignment function $q(\mathbf{x}_i)$ and leaf scores $w_1,\ldots,w_T$, observation $i$ receives score $w_{q(\mathbf{x}_i)}$. Define the summed gradient and Hessian in leaf $j$ as $G_j=\sum_{i:q(\mathbf{x}_i)=j}g_i$ and $H_j=\sum_{i:q(\mathbf{x}_i)=j}h_i$. Ignoring terms constant with respect to the new tree, and omitting the optional L1 term for this derivation, the approximate objective is:

$$
\widetilde{\mathcal{L}} = \sum_{j=1}^{T}\left[G_jw_j+\frac{1}{2}(H_j+\lambda)w_j^2\right]+\gamma T
$$

This quadratic form makes it possible to compute the best score for each leaf and evaluate whether a proposed split is worth its added complexity.

## Optimal Leaf Scores and Split Gain

For a fixed tree structure and $\alpha=0$, minimize the approximate objective independently for each leaf. Setting the derivative with respect to $w_j$ to zero gives the optimal leaf score:

$$
w_j^* = -\frac{G_j}{H_j+\lambda}
$$

Substituting these scores back into the objective gives the structure score, up to a constant:

$$
\mathrm{Score} = -\frac{1}{2}\sum_{j=1}^{T}\frac{G_j^2}{H_j+\lambda}+\gamma T
$$

A candidate split replaces one parent leaf with left and right child leaves. Its gain is the reduction in the approximate objective:

$$
\mathrm{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}-\frac{G^2}{H+\lambda}\right]-\gamma
$$

Here $G=G_L+G_R$ and $H=H_L+H_R$. A split is useful when it has positive gain and satisfies any other constraints. The $\gamma$ term charges for creating the extra leaf. With L1 regularization, the optimal leaf score uses soft-thresholding: $w_j^*=-\operatorname{sgn}(G_j)\max(|G_j|-\alpha,0)/(H_j+\lambda)$.

## Worked Example: Why a Split Helps

Use the squared-error loss $\ell(y,p)=\frac{1}{2}(y-p)^2$, whose gradient is $g=p-y$ and Hessian is $h=1$. Suppose the current prediction is $p=0$ for two observations with targets 2 and 4. For simplicity, set $\lambda=0$.

Without a split, the leaf has $G=-6$ and $H=2$, so its optimal score is $w^*=-G/H=3$. If a candidate split puts one observation in each child, the left leaf has $G_L=-2,H_L=1$ and the right has $G_R=-4,H_R=1$. Their optimal scores are 2 and 4, matching their targets.

The split gain is:

$$
\mathrm{Gain}=\frac{1}{2}\left[\frac{(-2)^2}{1}+\frac{(-4)^2}{1}-\frac{(-6)^2}{2}\right]-\gamma = 1-\gamma
$$

With $\gamma=0$, the split improves the objective by 1. If $\gamma\geq1$, this split is not worthwhile under this simplified calculation. In practice, other constraints such as minimum child weight can also reject a split.

## Complete Boosting Round: Data to Updated Predictions

Use the half-squared-error loss $\ell(y,p)=\frac{1}{2}(y-p)^2$ on two rows:

| Row | Feature $x$ | Target $y$ | Current prediction $p$ |
|---|---:|---:|---:|
| A | 0 | 0 | 1 |
| B | 1 | 2 | 1 |

For this loss, $g=p-y$ and $h=1$. Thus row A has $(g,h)=(1,1)$ and row B has $(g,h)=(-1,1)$. Consider a candidate split at $x\leq0.5$. The parent sums are $G=0,H=2$; the left sums are $G_L=1,H_L=1$; the right sums are $G_R=-1,H_R=1$.

With $\lambda=0$ and $\gamma=0$, its split gain is:

$$
\mathrm{Gain}=\frac{1}{2}\left[\frac{1^2}{1}+\frac{(-1)^2}{1}-\frac{0^2}{2}\right]=1
$$

The split is beneficial. Its optimal leaf weights are $w_L^*=-G_L/H_L=-1$ and $w_R^*=-G_R/H_R=1$. With learning rate $\eta=0.5$, the new predictions are:

$$
p_A^{new}=1+0.5(-1)=0.5,
\qquad
p_B^{new}=1+0.5(1)=1.5.
$$

Before the tree, the total half-squared loss is $1$. After the update it is $\frac{1}{2}(0-0.5)^2+\frac{1}{2}(2-1.5)^2=0.25$. This single round follows the full chain: current predictions, gradient/Hessian sums, split scoring, leaf weights, shrinkage, and new predictions.

## Revisit the Split with Regularization

Reuse the two-row example above, so the parent has $G=0,H=2$ and each child has $|G|=1,H=1$. Without regularization, $\lambda=0,\gamma=0$, the optimal leaf weights are $-1$ and $+1$, and the split gain is 1.

Now set $\lambda=1$ and $\gamma=0.2$. The L2 penalty changes each optimal leaf weight to:

$$
w_L^*=-\frac{1}{1+1}=-0.5,
\qquad
w_R^*=\frac{1}{1+1}=0.5.
$$

The regularized split gain is:

$$
\mathrm{Gain}=\frac{1}{2}\left[\frac{1}{2}+\frac{1}{2}-0\right]-0.2=0.3
$$

The split still has positive gain, but its score drops from 1 to 0.3. Here $\lambda$ shrinks the leaf values, while $\gamma$ charges for adding the split and can reject it if the adjusted gain is not positive. With $\eta=0.5$, the new predictions are $0.75$ and $1.25$ instead of $0.5$ and $1.5$ from the unregularized example.

## Important Parameters

Parameters control the number of boosting rounds, each tree's complexity, and the strength of regularization. Tune them using a validation strategy that matches the data and task.

| Parameter | Role | Typical effect of increasing it |
|---|---|---|
| `learning_rate` (`eta`) | Shrinks each tree's contribution | Slower, more conservative updates; often paired with more trees |
| `n_estimators` / boosting rounds | Number of sequential trees | More opportunities to improve fit, with more cost and overfitting risk |
| `max_depth` or `max_leaves` | Limits individual tree complexity | More detailed interactions, with increased variance and cost |
| `min_child_weight` | Minimum summed Hessian required in a child | More conservative splits; this is not simply a minimum row count |
| `gamma` (`min_split_loss`) | Minimum gain required to add a split | Fewer splits |
| `reg_lambda` / `lambda` | L2 penalty on leaf scores | Smaller leaf scores and stronger regularization |
| `reg_alpha` / `alpha` | L1 penalty on leaf scores | More shrinkage and potentially zero-valued leaf scores |
| `subsample` | Fraction of rows sampled per boosting round | More sampling randomness; values below 1 can reduce overfitting |
| `colsample_bytree` and related options | Fraction of features sampled | More feature randomness; may improve generalization |

Other settings matter too: `objective` defines the training loss, and `eval_metric` reports a metric for monitoring. `scale_pos_weight` can adjust positive-class weighting in some imbalanced binary tasks, but it does not replace choosing an appropriate evaluation metric or decision threshold. A larger model is not automatically better; use validation and early stopping where appropriate.

XGBoost provides tree-building methods such as `exact`, `approx`, and `hist`. Histogram-based methods bucket feature values to reduce split-search cost and are commonly used for larger datasets. Available parameters and defaults can vary by XGBoost version and API.

## Sparse Data and Missing Values

XGBoost includes sparsity-aware split search. For a candidate split, it can evaluate how missing values should be routed and learn a default direction that gives the better objective. This lets a tree use missingness as part of its split behavior without requiring every missing numeric value to be imputed first.

This does not mean missing values are always harmless. Missingness may reflect a meaningful process or a data-quality issue, and the learned routing can change across splits. Understand why values are missing and compare preprocessing choices using training-only fits and validation data.

## When to Use XGBoost

XGBoost is a strong candidate for structured or tabular data when nonlinear relationships and feature interactions may matter. Tree-based models generally do not require features to be standardized, and XGBoost can often handle missing numeric values through learned default split directions.

Keep several trade-offs in view:

- It can require substantial tuning and compute, especially with many rounds, deep trees, or large search spaces.
- Flexible trees can overfit; regularization, subsampling, validation, and early stopping help control this risk.
- Tree ensembles are less directly interpretable than a small decision tree or a linear model. Feature-importance summaries describe model behavior imperfectly and should not be treated as causal explanations.
- XGBoost is not inherently best for every dataset. Compare it against simple baselines and other suitable algorithms.

Choose the objective to match the task, create train/validation/test splits before fitting data-dependent preprocessing, and monitor a metric aligned with the real goal. Use early stopping against validation data rather than the test set. For classification, separately choose a decision threshold that reflects the costs of false positives and false negatives.

The central idea is that each tree is judged not only by how much it reduces loss, but also by the complexity it adds. Gradients and Hessians make candidate tree structures efficient to score; regularization helps control the resulting model.